In [4]:
import crypten
import torch

import numpy as np

# 初始化 Crypten
crypten.init()

/home/pai/lib/python3.9/site-packages/crypten/__init__.py:64: RuntimeWarning: CrypTen is already initialized.
  warnings.warn("CrypTen is already initialized.", RuntimeWarning)


In [23]:
def approx_relu(x_enc):
    # step0：实际由可信第三方TTP生成
    R1, R2 = 2, 1
    R1_rec = 1 / R1
    b = np.sqrt(R2 / R1**2)
    
    R1 = crypten.cryptensor(R1,device=x_enc.device)
    R1_rec = crypten.cryptensor(R1_rec,device=x_enc.device)
    R2 = crypten.cryptensor(R2,device=x_enc.device)
    b = crypten.cryptensor(b,device=x_enc.device)
        
    # step1
    xR1 = x_enc * R1
    y = x_enc.square() * R1.square() + R2
    # y = y.detach()
    y = y.get_plain_text()
        
    
    # step2
    # a = torch.sqrt(y)
    a = y.sqrt()
    
    # step3
    z = (xR1 + a) / 2
    
    # step4
    z = z*R1_rec - 0.5*b

    return z

In [44]:
# 创建输入数据和目标数据（示例中的示意数据）
x_plain = torch.tensor([1.0, 2.0, 3.0])
x_enc = crypten.cryptensor(x_plain,requires_grad=True)



# 调用 relu 近似函数
z_enc = approx_relu(x_enc)
x = z_enc.get_plain_text()

# 打印结果（密文形式）
print("Result (encrypted):", z_enc.get_plain_text())

z_enc = z_enc.sum()

# 进行反向传播
z_enc.backward()

# 打印梯度（密文形式）
print("Gradient (encrypted):", x_enc.grad.get_plain_text())

Result (encrypted): tensor([0.8089, 1.7807, 2.7710])
Gradient (encrypted): tensor([0.9471, 0.9849, 0.9933])


In [46]:
R1, R2 = 2, 1
0.5 + x / (2 * torch.sqrt(x**2 + R2/R1**2))

tensor([0.9253, 0.9814, 0.9921])